# PhonePe Pulse — Silver File Metadata

This notebook converts S3 source paths into structured metadata.

It extracts:
- Dataset group
- Business domain
- Country/state file scope
- State slug
- Year and quarter
- Reproducible period identifier

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "workspace.bronze.phonepe_raw_json"
SILVER_TABLE = "workspace.silver.phonepe_file_metadata"
EXPECTED_FILE_COUNT = 11_184

bronze_df = spark.table(BRONZE_TABLE)
bronze_count = bronze_df.count()

print(f"Bronze source table : {BRONZE_TABLE}")
print(f"Silver target table : {SILVER_TABLE}")
print(f"Bronze rows         : {bronze_count:,}")

assert bronze_count == EXPECTED_FILE_COUNT
print("Bronze source validation passed.")

Bronze source table : workspace.bronze.phonepe_raw_json
Silver target table : workspace.silver.phonepe_file_metadata
Bronze rows         : 11,184
Bronze source validation passed.


In [0]:
state_slug_from_path = F.regexp_extract(
    F.col("source_path"),
    r"/state/([^/]+)/\d{4}/[1-4]\.json$",
    1,
)

metadata_df = (
    bronze_df
    .select(
        "source_path",
        "source_modified_at",
        "source_size_bytes",
        "source_commit",
        "ingested_at_utc",
        "content_sha256",
    )
    .withColumn(
        "relative_path",
        F.regexp_extract("source_path", r"/data/(.+)$", 1),
    )
    .withColumn(
        "dataset_group",
        F.regexp_extract("source_path", r"/data/([^/]+)/", 1),
    )
    .withColumn(
        "domain",
        F.regexp_extract("source_path", r"/data/[^/]+/([^/]+)/", 1),
    )
    .withColumn(
        "file_scope",
        F.when(state_slug_from_path != "", F.lit("state"))
        .otherwise(F.lit("country")),
    )
    .withColumn(
        "state_slug",
        F.when(state_slug_from_path != "", state_slug_from_path)
        .otherwise(F.lit(None).cast("string")),
    )
    .withColumn(
        "year",
        F.regexp_extract(
            "source_path",
            r"/(\d{4})/[1-4]\.json$",
            1,
        ).cast("int"),
    )
    .withColumn(
        "quarter",
        F.regexp_extract(
            "source_path",
            r"/\d{4}/([1-4])\.json$",
            1,
        ).cast("int"),
    )
    .withColumn(
        "period",
        F.concat(
            F.col("year").cast("string"),
            F.lit("-Q"),
            F.col("quarter").cast("string"),
        ),
    )
)

invalid_metadata_df = metadata_df.filter(
    (~F.col("dataset_group").isin("aggregated", "map", "top"))
    | (~F.col("domain").isin("merchant", "transaction", "user"))
    | (~F.col("file_scope").isin("country", "state"))
    | F.col("year").isNull()
    | F.col("quarter").isNull()
    | (
        (F.col("file_scope") == "state")
        & F.col("state_slug").isNull()
    )
)

invalid_metadata_count = invalid_metadata_df.count()

print(f"Metadata rows        : {metadata_df.count():,}")
print(f"Invalid metadata rows: {invalid_metadata_count:,}")

assert invalid_metadata_count == 0
print("Path metadata extraction passed.")

Metadata rows        : 11,184
Invalid metadata rows: 0
Path metadata extraction passed.


In [0]:
coverage_df = (
    metadata_df
    .groupBy("dataset_group", "domain", "file_scope")
    .agg(
        F.min("period").alias("first_period"),
        F.max("period").alias("last_period"),
        F.count("*").alias("file_count"),
        F.countDistinct("state_slug").alias("unique_states"),
    )
    .orderBy("dataset_group", "domain", "file_scope")
)

unexpected_coverage_count = coverage_df.filter(
    (
        (F.col("file_scope") == "country")
        & (F.col("file_count") != 34)
    )
    | (
        (F.col("file_scope") == "state")
        & (F.col("domain") == "merchant")
        & (F.col("file_count") != 1178)
    )
    | (
        (F.col("file_scope") == "state")
        & (F.col("domain").isin("transaction", "user"))
        & (F.col("file_count") != 1224)
    )
).count()

display(coverage_df)

assert unexpected_coverage_count == 0
print("Dataset coverage validation passed.")

dataset_group,domain,file_scope,first_period,last_period,file_count,unique_states
aggregated,merchant,country,2018-Q1,2026-Q2,34,0
aggregated,merchant,state,2018-Q1,2026-Q2,1178,36
aggregated,transaction,country,2018-Q1,2026-Q2,34,0
aggregated,transaction,state,2018-Q1,2026-Q2,1224,36
aggregated,user,country,2018-Q1,2026-Q2,34,0
aggregated,user,state,2018-Q1,2026-Q2,1224,36
map,merchant,country,2018-Q1,2026-Q2,34,0
map,merchant,state,2018-Q1,2026-Q2,1178,36
map,transaction,country,2018-Q1,2026-Q2,34,0
map,transaction,state,2018-Q1,2026-Q2,1224,36


Dataset coverage validation passed.


In [0]:
(
    metadata_df
    .repartition(4)
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

spark.sql(
    f"""
    ALTER TABLE {SILVER_TABLE}
    SET TBLPROPERTIES (
        'data_layer' = 'silver',
        'source_system' = 'phonepe_pulse',
        'description' = 'Structured metadata extracted from PhonePe source file paths'
    )
    """
)

silver_count = spark.table(SILVER_TABLE).count()

print(f"Silver table: {SILVER_TABLE}")
print(f"Rows written: {silver_count:,}")

assert silver_count == EXPECTED_FILE_COUNT
print("Silver metadata table validation passed.")

Silver table: workspace.silver.phonepe_file_metadata
Rows written: 11,184
Silver metadata table validation passed.
